# Scale Dependence Study: l=4 vs l=6

Analyze how p-adic embedding structure changes across hierarchical scales.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

cache_dir = Path('/Volumes/Fangorn/padic_fractal_analysis/cache')

with open(cache_dir / 'fractal_metrics.json') as f:
    metrics = json.load(f)

print("Scale-Dependent Analysis")

## Hierarchical Scaling Properties

In [ ]:
l4 = metrics['l=4']
l6 = metrics['l=6']

# Scale factors
scale_ratio = l6['scale_info']['n_regions'] / l4['scale_info']['n_regions']
nn_dist_scaling = l6['clustering']['mean_nn_distance'] / l4['clustering']['mean_nn_distance']
spread_scaling = l6['spreading']['embedding_span'] / l4['spreading']['embedding_span']
density_cv_scaling = l6['sierpinski_properties']['density_cv'] / l4['sierpinski_properties']['density_cv']

print(f"\nScaling Analysis (l=4 → l=6):")
print(f"  Region count scaling: {scale_ratio:.1f}x")
print(f"  NN distance scaling: {nn_dist_scaling:.4f}")
print(f"  Embedding spread scaling: {spread_scaling:.4f}")
print(f"  Density CV scaling: {density_cv_scaling:.4f}")
print(f"\nInterpretation:")
print(f"  - NN distance <1: Finer scale shows tighter clustering (fractal)")
print(f"  - Spread ~1: Sierpinski structure maintains same extent")
print(f"  - Density CV <1: Structure becomes more uniform at finer scales")

## Comparative Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

scales = ['l=4', 'l=6']
for idx, scale_name in enumerate(scales):
    embeddings = np.load(str(cache_dir / f'padic_embeddings_{scale_name}.npy'))
    grid_values = np.load(str(cache_dir / f'dem_grid_{scale_name}.npy'))
    
    # Top: Embedding space
    ax = axes[0, idx]
    scatter = ax.scatter(embeddings[:, 0], embeddings[:, 1],
                         c=grid_values.flatten(), cmap='viridis', s=80,
                         alpha=0.7, edgecolors='black', linewidth=0.5)
    ax.set_title(f'{scale_name}: Embedding Space', fontweight='bold')
    ax.set_xlabel('Real')
    ax.set_ylabel('Imag')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    
    # Bottom: Clustering distribution
    ax = axes[1, idx]
    distances_from_origin = np.sqrt(embeddings[:, 0]**2 + embeddings[:, 1]**2)
    ax.hist(distances_from_origin, bins=20, alpha=0.7, edgecolor='black')
    ax.set_title(f'{scale_name}: Distance Distribution', fontweight='bold')
    ax.set_xlabel('Distance from Origin')
    ax.set_ylabel('Count')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Self-Similarity Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NN distance comparison
nn_l4 = l4['clustering']['mean_nn_distance']
nn_l6 = l6['clustering']['mean_nn_distance']

ax = axes[0]
x = [0, 1]
y = [nn_l4, nn_l6]
ax.plot(x, y, 'o-', linewidth=2, markersize=10)
ax.set_xticks(x)
ax.set_xticklabels(['l=4', 'l=6'])
ax.set_ylabel('Mean NN Distance')
ax.set_title('Nearest-Neighbor Distance Scaling', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.text(0.5, nn_l4, f'  {nn_l4:.4f}', va='center')
ax.text(0.5, nn_l6, f'  {nn_l6:.4f}', va='center')

# Density CV comparison
cv_l4 = l4['sierpinski_properties']['density_cv']
cv_l6 = l6['sierpinski_properties']['density_cv']

ax = axes[1]
x = [0, 1]
y = [cv_l4, cv_l6]
ax.plot(x, y, 's-', linewidth=2, markersize=10, color='orange')
ax.set_xticks(x)
ax.set_xticklabels(['l=4', 'l=6'])
ax.set_ylabel('Density Coefficient of Variation')
ax.set_title('Clustering Tendency Across Scales', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.text(0.5, cv_l4, f'  {cv_l4:.4f}', va='center')
ax.text(0.5, cv_l6, f'  {cv_l6:.4f}', va='center')

plt.tight_layout()
plt.show()

print(f"\nFractal Dimension Indicators:")
print(f"  NN distance scaling of {nn_dist_scaling:.4f} suggests fractal properties")
print(f"  Expected for uniform distribution: 1/{np.sqrt(scale_ratio):.4f}")
print(f"  Observed scaling indicates {'moderate' if nn_dist_scaling < 0.3 else 'weak'} self-similarity")